# 第25章：Triton 编译器 IR — 从 Python 到 PTX

## 前置知识
- 完成 Part 1-3

## 学习目标
- 理解 Triton 的编译流水线
- 学会检查各级 IR（中间表示）
- 理解编译器优化对性能的影响

In [1]:
import torch
import triton
import triton.language as tl

## 25.1 Triton 编译流水线

```
Python 代码 (@triton.jit)
       │
       ▼
┌─────────────┐
│  Triton AST  │  Python AST → Triton 自定义 AST
└──────┬──────┘
       │
       ▼
┌─────────────┐
│  Triton IR   │  TTIR: 高级 IR，保留块级语义
│  (TTIR)      │  - tl.load → tt.load
│              │  - tl.dot → tt.dot
└──────┬──────┘
       │  优化: 常量折叠, 死代码消除
       ▼
┌─────────────┐
│  Triton GPU  │  TTGIR: GPU 特定 IR
│  IR (TTGIR)  │  - 插入共享内存分配
│              │  - 确定 warp 调度
│              │  - 软件流水线
└──────┬──────┘
       │  lowering
       ▼
┌─────────────┐
│  LLVM IR     │  标准 LLVM IR
│              │  - 向量化
│              │  - 寄存器分配
└──────┬──────┘
       │  LLVM backend
       ▼
┌─────────────┐
│  PTX        │  NVIDIA GPU 汇编
│              │  - mma.sync (Tensor Core)
│              │  - cp.async (异步拷贝)
│              │  - ldmatrix
└──────┬──────┘
       │  ptxas
       ▼
┌─────────────┐
│  CUBIN      │  GPU 可执行二进制
└─────────────┘
```

## 25.2 查看编译产物

Triton 编译后的文件缓存在 `~/.triton/cache/` 中。

### 环境变量控制

```bash
# 打印 TTIR
MLIR_ENABLE_DUMP=1 python script.py

# 保存所有中间 IR
TRITON_CACHE_DIR=/tmp/triton_cache python script.py

# 禁用缓存（总是重新编译）
TRITON_CACHE_DIR="" python script.py
```

In [2]:
# 一个简单的 kernel 用于演示 IR 检查
@triton.jit
def simple_add_kernel(a_ptr, b_ptr, c_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    a = tl.load(a_ptr + offs, mask=mask)
    b = tl.load(b_ptr + offs, mask=mask)
    tl.store(c_ptr + offs, a + b, mask=mask)

# 触发编译
n = 1024
a = torch.randn(n, device='cuda')
b = torch.randn(n, device='cuda')
c = torch.empty_like(a)
simple_add_kernel[(1,)](a, b, c, n, BLOCK=1024)
print(f"编译完成，结果正确: {torch.allclose(c, a + b)}")

编译完成，结果正确: True


In [3]:
# 查看编译后的汇编信息
# 获取编译后 kernel 的元信息
print("=== Kernel 编译信息 ===")

# 查看 Triton 缓存目录
import os
cache_dir = os.environ.get("TRITON_CACHE_DIR", os.path.join(os.path.expanduser("~"), ".triton", "cache"))
print(f"Triton 缓存目录: {cache_dir}")

# 检查 GEMM kernel 使用了多少寄存器和共享内存
@triton.jit
def matmul_for_ir(a_ptr, b_ptr, c_ptr, M, N, K,
                  stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                  BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        a_offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        a_offs_k = k + tl.arange(0, BLOCK_K)
        a = tl.load(a_ptr + a_offs_m[:, None] * stride_am + a_offs_k[None, :] * stride_ak,
                    mask=(a_offs_m[:, None] < M) & (a_offs_k[None, :] < K), other=0.0)
        b_offs_k = k + tl.arange(0, BLOCK_K)
        b_offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        b = tl.load(b_ptr + b_offs_k[:, None] * stride_bk + b_offs_n[None, :] * stride_bn,
                    mask=(b_offs_k[:, None] < K) & (b_offs_n[None, :] < N), other=0.0)
        acc += tl.dot(a, b)
    c_offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    c_offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    tl.store(c_ptr + c_offs_m[:, None] * stride_cm + c_offs_n[None, :] * stride_cn,
             acc.to(tl.float16),
             mask=(c_offs_m[:, None] < M) & (c_offs_n[None, :] < N))

# 触发编译
M, N, K = 256, 256, 128
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)
c = torch.empty(M, N, device='cuda', dtype=torch.float16)
matmul_for_ir[(4, 4)](a, b, c, M, N, K,
                       a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1),
                       BLOCK_M=64, BLOCK_N=64, BLOCK_K=32)
print(f"GEMM 编译完成，正确性: {torch.allclose(c.float(), (a.float() @ b.float()), atol=0.5)}")

=== Kernel 编译信息 ===
Triton 缓存目录: /root/.triton/cache
GEMM 编译完成，正确性: True


## 25.3 编译器优化详解

Triton 编译器自动进行的关键优化：

### 1. 自动共享内存管理
```
你写的:          编译器生成的:
a = tl.load(...)  →  __shared__ a_smem[...]
                     cp.async(..., a_smem)  // 异步加载到共享内存
                     __syncthreads()
                     reg_a = a_smem[...]    // 从共享内存加载到寄存器
```

### 2. 自动向量化
```
你写的:           编译器生成的:
tl.load(ptr + offs)  →  ld.global.v4.f16 {r0,r1,r2,r3}, [addr]
                        // 4个 fp16 合并为一次 128-bit 加载
```

### 3. 软件流水线
```
你写的:              编译器生成的 (num_stages=3):
for k in range(K):    stage 0: load k+2
  a = load(...)       stage 1: load k+1 (from smem to reg)
  b = load(...)       stage 2: compute k (mma.sync)
  acc += dot(a,b)     // 三级流水线重叠执行
```

### 4. Tensor Core 映射
```
你写的:           编译器生成的:
tl.dot(a, b)  →  ldmatrix.sync.aligned.m8n8.x4 ...
                  mma.sync.aligned.m16n8k16.row.col.f32.f16.f16.f32 ...
```

## 25.4 理解 PTX 输出

关键的 PTX 指令：

| PTX 指令 | 含义 | Triton 来源 |
|----------|------|------------|
| `ld.global.v4` | 向量化全局内存加载 | `tl.load` |
| `st.global.v4` | 向量化全局内存存储 | `tl.store` |
| `ld.shared` | 共享内存加载 | 自动生成 |
| `cp.async` | 异步内存拷贝 (Ampere+) | `num_stages` |
| `mma.sync` | Tensor Core 矩阵乘 | `tl.dot` |
| `ldmatrix` | 高效矩阵加载 | `tl.dot` |
| `red.global.add` | 全局原子加 | `tl.atomic_add` |
| `bar.sync` | 屏障同步 | 自动生成 |

### 如何查看 PTX
```python
# Triton kernel 编译后，PTX 存储在缓存中
# 位置: ~/.triton/cache/<hash>/<kernel_name>.ptx

# 或者使用 CUDA:
# cuobjdump -ptx kernel.cubin
```

## 总结

```
编译流水线:
Python → TTIR → TTGIR → LLVM IR → PTX → CUBIN

关键编译器优化:
├── 共享内存分配和管理
├── 向量化加载/存储
├── 软件流水线 (num_stages)
├── Tensor Core 指令映射
├── 寄存器分配
└── 内存合并访问
```

理解编译器的工作有助于：
1. 写出对编译器更友好的代码
2. 理解为什么某些配置比其他更快
3. 调试性能问题

## 练习

1. 找到你系统上的 Triton 缓存目录，查看编译产物
2. 对比 BLOCK_SIZE=64 和 BLOCK_SIZE=128 的 GEMM PTX 差异
3. 计算一个 GEMM kernel 的理论共享内存使用量